# FGSM move classifier: all characters, simulated match footage

**Doesn't use Google Drive** (yours is full). Checkpoints and the dataset cache go to your S3 bucket instead.

**Open it without uploading** (uploading saves to Drive and fails when Drive is full):
colab.research.google.com → File → Open notebook → **GitHub** tab → sign in to GitHub → repo `lionelishi21/fgsm-vision-engine` → `notebooks/All_Characters_Training.ipynb`.

Then **Runtime → Change runtime type → T4 GPU**, and run the cells top to bottom.

Before you start, generate a download link for the code package on your Mac (valid 1 hour):
```
aws s3 presign s3://fgsm-vision-models-aibridix-official/fgsm_allchars_code.zip --expires-in 3600 --profile aibridix_official
```

**If Colab disconnects:** reconnect and run every cell again from the top. The dataset comes back from S3 (no re-scrape) and training resumes from the last checkpoint synced to S3 (every 5 minutes).

In [ ]:
# 1. Check the GPU is attached - training on CPU would take days.
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then run this again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. AWS access for the checkpoint + dataset storage in S3 (same keys as the
# aibridix_official profile on your Mac: ~/.aws/credentials). Entered here, never saved.
import getpass, os
os.environ['AWS_ACCESS_KEY_ID'] = getpass.getpass('AWS Access Key ID: ')
os.environ['AWS_SECRET_ACCESS_KEY'] = getpass.getpass('AWS Secret Access Key: ')
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
!pip install -q awscli
!aws s3 ls s3://fgsm-vision-models-aibridix-official/ > /dev/null && echo 'S3 access OK'

In [ ]:
# 3. Download the code + the 5-character model to start from. Paste the presigned URL when asked.
%cd /content
from getpass import getpass
code_url = getpass('Presigned S3 URL for fgsm_allchars_code.zip: ')
!wget -q -O fgsm_allchars_code.zip "{code_url}"
!rm -rf /content/fgsm_allchars && unzip -o -q fgsm_allchars_code.zip -d /content/fgsm_allchars
%cd /content/fgsm_allchars
!ls

In [ ]:
# 4. Install dependencies. torch/torchvision are skipped on purpose: Colab's own
# CUDA build must stay, and upgrading it from requirements.txt breaks the GPU.
!grep -vE '^(torch|torchvision)([=<>].*)?$' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# 5. Dataset: every SF6 character's move clips from ultimateframedata.com.
# First run scrapes (~15 min, ~6-7 GB) and uploads a cache to S3; later runs download it.
import subprocess
BUCKET = 's3://fgsm-vision-models-aibridix-official'
CACHE = f'{BUCKET}/datasets/ufd_all_characters.tar'
cached = subprocess.run(['aws', 's3', 'ls', CACHE], capture_output=True).returncode == 0
if cached:
    print('Restoring dataset from', CACHE)
    !aws s3 cp {CACHE} /content/ufd_all_characters.tar --only-show-errors
    !tar -xf /content/ufd_all_characters.tar -C /content/fgsm_allchars && rm /content/ufd_all_characters.tar
else:
    !python src/scripts/scrape_ufd_gifs.py
    !tar -cf /content/ufd_all_characters.tar data/ufd
    !aws s3 cp /content/ufd_all_characters.tar {CACHE} --only-show-errors && rm /content/ufd_all_characters.tar && echo 'Cached dataset to S3'
!du -sh data/ufd
!echo "characters: $(ls data/ufd/move_classification_dataset | wc -l)"

In [ ]:
# 6. Train. Checkpoints are written locally and synced to S3 every 5 minutes;
# on a re-run, the latest ones are pulled back from S3 first so training resumes.
#
# Colab's torch JIT-compiles a kernel on the first step and has crashed with
# "nvrtc: failed to open libnvrtc-builtins.so" when that library isn't on the
# loader path, so put it there first.
import glob, os, subprocess, sys

def _find_nvrtc_builtins():
    return glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*', recursive=True)

hits = _find_nvrtc_builtins()
if not hits:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', 'nvidia-cuda-nvrtc==13.0.88'], check=False)
    hits = _find_nvrtc_builtins()
print('nvrtc builtins:', hits)
if hits:
    os.environ['LD_LIBRARY_PATH'] = os.path.dirname(hits[0]) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

OUT = '/content/runs/temporal_move_classifier_all_characters'
REMOTE = 's3://fgsm-vision-models-aibridix-official/colab_runs/temporal_move_classifier_all_characters'
!mkdir -p {OUT}
!aws s3 sync {REMOTE}/ {OUT}/ --only-show-errors
!ls {OUT}
get_ipython().system_raw(f'while true; do sleep 300; aws s3 sync {OUT}/ {REMOTE}/ --delete --quiet; done &')

!sed -i 's#output_dir:.*#output_dir: {OUT}#' configs/temporal_classifier_all_characters.yaml
!python src/train_temporal_classifier.py --config configs/temporal_classifier_all_characters.yaml
!aws s3 sync {OUT}/ {REMOTE}/ --delete --only-show-errors && echo 'Final model synced to S3'

In [ ]:
# 7. Check the new model on the same probes as the 5-character model
# (runs/generalization/probes_base.json). Compare the "match-scale camera",
# "16 consecutive frames" and "all combined" rows - those collapsed before.
!python src/evaluate_generalization.py probes \
    --model-dir /content/runs/temporal_move_classifier_all_characters \
    --data-dir data/ufd \
    --json-out /content/runs/probes_all_characters.json
!aws s3 cp /content/runs/probes_all_characters.json s3://fgsm-vision-models-aibridix-official/colab_runs/probes_all_characters.json